In [6]:
import os
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")          # non-interactive backend for file output
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms


# ─────────────────────────────────────────────
# Part 1: PrunableLinear Layer
# ─────────────────────────────────────────────

class PrunableLinear(nn.Module):
    """
    A drop-in replacement for nn.Linear that augments each weight with a
    learnable scalar gate in [0, 1].

    How it works
    ─────────────
    For every weight w_ij we introduce a companion score s_ij (gate_scores).
    During the forward pass:

        gate_ij  = sigmoid(s_ij)          ∈ (0, 1)
        w̃_ij    = w_ij * gate_ij          (pruned weight)
        y = W̃ x + b                       (standard linear op)

    When gate_ij → 0  the connection is effectively removed.
    When gate_ij → 1  the connection passes through unchanged.

    Gradient flow
    ─────────────
    Both weight and gate_scores receive gradients through the standard
    autograd graph because pruned_weights = weight * sigmoid(gate_scores)
    is a differentiable expression.  The sparsity regularisation term
    pushes gate_scores towards -∞, which drives sigmoid(gate_scores) → 0.

    Parameters
    ──────────
    in_features  : int  – size of each input sample
    out_features : int  – size of each output sample
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        # ── Standard parameters ────────────────────────────────────────────
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.empty(out_features))

        # ── Gate scores (same shape as weight) ────────────────────────────
        # Registered as a Parameter so the optimiser updates them alongside
        # the weights.  Initialised to 3.0 so sigmoid(3.0) ≈ 0.95, meaning
        # all gates start nearly open — the network learns which ones to close.
        self.gate_scores = nn.Parameter(torch.empty(out_features, in_features))

        self._reset_parameters()

    def _reset_parameters(self):
        # Kaiming uniform for weights (standard for ReLU networks)
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

        # Zero bias (standard)
        nn.init.zeros_(self.bias)

        # High initial gate scores → gates ≈ open
        nn.init.constant_(self.gate_scores, 3.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass with gated weights.

        Steps
        ─────
        1. gates         = sigmoid(gate_scores)   shape: [out, in]
        2. pruned_weights = weight * gates         element-wise product
        3. output         = x @ pruned_weights.T + bias
        """
        # Step 1 – transform scores to (0, 1)
        gates = torch.sigmoid(self.gate_scores)

        # Step 2 – element-wise mask on weights
        pruned_weights = self.weight * gates

        # Step 3 – standard linear operation
        return F.linear(x, pruned_weights, self.bias)

    def sparsity_loss(self) -> torch.Tensor:
        """
        L1 penalty for this layer: Σ sigmoid(gate_scores).

        Why L1?
        ───────
        The L1 norm (sum of absolute values) is the tightest convex relaxation
        of the L0 "count of non-zeros" norm.  It applies a CONSTANT gradient
        of ±1 regardless of gate magnitude, which uniquely pushes small values
        all the way to exactly zero rather than just shrinking them (as L2
        would do).  Since sigmoid outputs are strictly positive, the sum equals
        the L1 norm directly — no abs() needed.
        """
        return torch.sum(torch.sigmoid(self.gate_scores))

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}"


# ─────────────────────────────────────────────
# Network definition
# ─────────────────────────────────────────────

class PrunableNet(nn.Module):
    """
    Feed-forward MLP for CIFAR-10 (10-class image classification).

    Architecture
    ─────────────
    Input : 32×32×3 image  → flatten → 3072-dim vector
    FC1   : 3072 → 512  + ReLU
    FC2   : 512  → 256  + ReLU
    FC3   : 256  → 10   (logits)

    All three linear layers are PrunableLinear, so every weight has a
    companion gate that can be driven to zero by the sparsity loss.
    """

    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = PrunableLinear(32 * 32 * 3, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)           # raw logits (CrossEntropyLoss handles softmax)

    def total_sparsity_loss(self) -> torch.Tensor:
        """Sum of L1 gate penalties across all prunable layers."""
        return self.fc1.sparsity_loss() + \
               self.fc2.sparsity_loss() + \
               self.fc3.sparsity_loss()

    def count_parameters(self) -> dict:
        """Return total weights and active gate counts (gate > 1e-2)."""
        total = 0
        active = 0
        for layer in [self.fc1, self.fc2, self.fc3]:
            gates = torch.sigmoid(layer.gate_scores).detach().cpu().numpy()
            total  += gates.size
            active += int((gates >= 1e-2).sum())
        return {"total": total, "active": active,
                "sparsity_pct": 100.0 * (1 - active / total)}


# ─────────────────────────────────────────────
# Data loading helpers
# ─────────────────────────────────────────────

def get_cifar10_loaders(batch_size: int = 128, data_root: str = "./data"):
    """
    Download (if needed) and return DataLoaders for CIFAR-10.

    Normalisation: mean=0.5, std=0.5 per channel (maps [0,1] → [-1,1]).
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5),
                             (0.5, 0.5, 0.5)),
    ])

    train_ds = torchvision.datasets.CIFAR10(
        root=data_root, train=True,  download=True, transform=transform)
    test_ds  = torchvision.datasets.CIFAR10(
        root=data_root, train=False, download=True, transform=transform)

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True)
    test_loader  = torch.utils.data.DataLoader(
        test_ds,  batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)

    return train_loader, test_loader


# ─────────────────────────────────────────────
# Part 3: Training and Evaluation
# ─────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer,
                    lambd: float, device: torch.device) -> tuple[float, float]:
    """
    Single training epoch.

    Returns
    ───────
    avg_class_loss   : float  – mean CrossEntropy over batches
    avg_total_loss   : float  – mean (ClassLoss + λ*SparsityLoss) over batches
    """
    model.train()
    total_cls_loss   = 0.0
    total_total_loss = 0.0
    n_batches        = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        # ── Forward ────────────────────────────────────────────────────────
        logits      = model(inputs)
        cls_loss    = criterion(logits, labels)

        # ── Sparsity regularisation ────────────────────────────────────────
        # Total Loss = CrossEntropyLoss + λ × Σ sigmoid(gate_scores)
        # The gradient of the sparsity term w.r.t. gate_scores pushes them
        # towards -∞, collapsing their sigmoid values towards 0.
        sp_loss     = model.total_sparsity_loss()
        total_loss  = cls_loss + lambd * sp_loss

        # ── Backward ───────────────────────────────────────────────────────
        # autograd differentiates through sigmoid + element-wise mul,
        # so gate_scores and weights both receive valid gradients.
        total_loss.backward()
        optimizer.step()

        total_cls_loss   += cls_loss.item()
        total_total_loss += total_loss.item()
        n_batches        += 1

    return total_cls_loss / n_batches, total_total_loss / n_batches


def evaluate(model, loader, device: torch.device) -> tuple[float, float, list]:
    """
    Evaluate on test set.

    Returns
    ───────
    accuracy       : float  – top-1 accuracy in %
    sparsity_pct   : float  – % weights with gate < 1e-2
    gate_values    : list   – all gate values (flattened, for histogram)
    """
    model.eval()
    correct = 0
    total   = 0
    all_gates = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs   = model(inputs)
            _, preds  = torch.max(outputs, 1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)

        for layer in [model.fc1, model.fc2, model.fc3]:
            g = torch.sigmoid(layer.gate_scores).cpu().numpy().flatten()
            all_gates.append(g)

    accuracy      = 100.0 * correct / total
    flat_gates    = np.concatenate(all_gates)
    sparsity_pct  = float(np.mean(flat_gates < 1e-2) * 100)

    return accuracy, sparsity_pct, flat_gates.tolist()


def train_and_evaluate(lambd: float, epochs: int = 10,
                        device: torch.device = None,
                        verbose: bool = True) -> dict:
    """
    Full training run for a given λ value.

    Parameters
    ──────────
    lambd   : sparsity regularisation coefficient
    epochs  : number of training epochs
    device  : torch device (auto-detected if None)
    verbose : print per-epoch progress

    Returns
    ───────
    dict with keys: lambd, accuracy, sparsity_pct, gate_values, history
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, test_loader = get_cifar10_loaders(batch_size=128)

    model     = PrunableNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    # Learning-rate schedule: decay by 0.5 at 60% and 80% of training
    scheduler = optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[int(epochs * 0.6), int(epochs * 0.8)],
        gamma=0.5,
    )
    criterion = nn.CrossEntropyLoss()

    history = {"cls_loss": [], "total_loss": []}

    print(f"\n{'─'*55}")
    print(f"  Training  λ = {lambd}   ({epochs} epochs)  device={device}")
    print(f"{'─'*55}")

    for epoch in range(1, epochs + 1):
        cls_l, tot_l = train_one_epoch(
            model, train_loader, criterion, optimizer, lambd, device)
        scheduler.step()

        history["cls_loss"].append(cls_l)
        history["total_loss"].append(tot_l)

        if verbose and (epoch % max(1, epochs // 5) == 0 or epoch == 1):
            print(f"  Epoch {epoch:3d}/{epochs}  "
                  f"cls_loss={cls_l:.4f}  total_loss={tot_l:.4f}")

    accuracy, sparsity_pct, gate_values = evaluate(model, test_loader, device)

    print(f"\n  ✓  λ={lambd} → Accuracy={accuracy:.2f}%  Sparsity={sparsity_pct:.1f}%")

    return {
        "lambd"       : lambd,
        "accuracy"    : accuracy,
        "sparsity_pct": sparsity_pct,
        "gate_values" : gate_values,
        "history"     : history,
    }


# ─────────────────────────────────────────────
# Visualisation helpers
# ─────────────────────────────────────────────

COLORS = {
    "low"    : "#4C72B0",   # blue
    "medium" : "#DD8452",   # orange
    "high"   : "#55A868",   # green
}

def plot_gate_distributions(results: list[dict], save_path: str = "gate_distributions.png"):
    """
    Plot histogram of final gate values for each λ experiment side-by-side.

    A successfully pruned network should show a tall spike at 0 (pruned
    connections) and a smaller cluster near 1 (kept connections).
    """
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), sharey=False)
    if n == 1:
        axes = [axes]

    color_list = list(COLORS.values())

    for ax, res, col in zip(axes, results, color_list):
        gates = np.array(res["gate_values"])

        ax.hist(gates, bins=80, color=col, alpha=0.85, edgecolor="white",
                linewidth=0.3)
        ax.axvline(x=1e-2, color="red", linestyle="--", linewidth=1.2,
                   label="prune threshold (0.01)")

        pct_zero = float(np.mean(gates < 1e-2) * 100)
        ax.set_title(
            f"λ = {res['lambd']}\n"
            f"Acc = {res['accuracy']:.1f}%  |  Sparsity = {res['sparsity_pct']:.1f}%",
            fontsize=11)
        ax.set_xlabel("Gate value (sigmoid output)", fontsize=9)
        ax.set_ylabel("Number of weights", fontsize=9)
        ax.legend(fontsize=8)
        ax.text(0.60, 0.88,
                f"{pct_zero:.1f}% weights < 0.01",
                transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow",
                          edgecolor="grey", alpha=0.9))
        ax.set_xlim(-0.02, 1.02)

    fig.suptitle("Distribution of Learned Gate Values Across All PrunableLinear Layers",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → Saved gate distribution plot to '{save_path}'")


def plot_training_curves(results: list[dict], save_path: str = "training_curves.png"):
    """
    Plot classification loss curves for all λ values.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    color_list = list(COLORS.values())

    for res, col in zip(results, color_list):
        lbl = f"λ={res['lambd']}"
        epochs = range(1, len(res["history"]["cls_loss"]) + 1)
        axes[0].plot(epochs, res["history"]["cls_loss"],
                     color=col, linewidth=2, label=lbl)
        axes[1].plot(epochs, res["history"]["total_loss"],
                     color=col, linewidth=2, label=lbl)

    for ax, title, ylabel in zip(
        axes,
        ["Classification Loss (CrossEntropy)", "Total Loss (CE + λ·Sparsity)"],
        ["Loss", "Loss"]
    ):
        ax.set_xlabel("Epoch", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → Saved training curves to '{save_path}'")


def plot_sparsity_accuracy_tradeoff(results: list[dict],
                                    save_path: str = "sparsity_accuracy_tradeoff.png"):
    """
    Scatter plot: sparsity level vs. test accuracy for each λ.
    """
    fig, ax = plt.subplots(figsize=(6, 4))
    color_list = list(COLORS.values())

    for res, col in zip(results, color_list):
        ax.scatter(res["sparsity_pct"], res["accuracy"],
                   s=120, color=col, zorder=5,
                   label=f"λ={res['lambd']}")
        ax.annotate(
            f"λ={res['lambd']}",
            (res["sparsity_pct"], res["accuracy"]),
            textcoords="offset points", xytext=(8, 4),
            fontsize=9
        )

    ax.set_xlabel("Sparsity Level (%)", fontsize=11)
    ax.set_ylabel("Test Accuracy (%)", fontsize=11)
    ax.set_title("Sparsity–Accuracy Trade-off", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  → Saved trade-off plot to '{save_path}'")


# ─────────────────────────────────────────────
# Report generation
# ─────────────────────────────────────────────

REPORT_TEMPLATE = r"""\
# Self-Pruning Neural Network — Results Report

## 1. Why L1 Penalty on Sigmoid Gates Encourages Sparsity

The training objective is:

$$\text{{Total Loss}} = \underbrace{{{\mathcal{{L}}_{{\text{{CE}}}}}}}_{{{\text{{classify correctly}}}}} + \lambda \cdot \underbrace{{{\sum_{{i,j}} \sigma(s_{{ij}})}}}_{{{\text{{SparsityLoss}}}}}$$

where $s_{{ij}}$ is the **gate score** for weight $w_{{ij}}$ and $\sigma(\cdot)$ is the sigmoid function.

### The mechanism

| Property | Effect |
|---|---|
| Sigmoid maps $s \in \mathbb{{R}}$ to $(0,1)$ | Gates are always bounded; no clipping needed |
| SparsityLoss = **L1 norm** of all gate values | Gradient is a **constant** $\lambda$ regardless of gate size |
| Constant gradient on small values | Pushes small gates all the way to **exactly 0** (unlike L2 which only shrinks them) |
| $\partial \text{{SparsityLoss}} / \partial s_{{ij}} = \lambda \cdot \sigma(s_{{ij}})(1-\sigma(s_{{ij}}))$ | Also flows back through sigmoid, nudging $s_{{ij}} \to -\infty$ |

When a gate $\sigma(s_{{ij}}) \approx 0$, the corresponding weight $w_{{ij}}$ is multiplied by ~0, effectively **removing** that connection from the network. The competing classification loss prevents *all* gates from being zeroed out — only those that contribute little to classification accuracy are pruned away.

---

## 2. Experimental Results

{results_table}

> **Interpretation:**
> A low λ barely prunes the network (classification loss dominates).
> A medium λ provides a good balance, retaining accuracy while removing many weak weights.
> A high λ aggressively prunes, compressing the model at the cost of some accuracy.

---

## 3. Gate Value Distribution

The histogram `gate_distributions.png` shows the distribution of all sigmoid gate values
after training. A successfully pruned model displays a characteristic **bimodal** pattern:

- **Tall spike at 0** — gates that were driven to near-zero (pruned connections)
- **Cluster near 1** — gates of important connections that were preserved

As λ increases, the spike at 0 grows taller, confirming stronger pruning.

---

## 4. Analysis of the λ Trade-off

{analysis}

---

*Generated automatically by `self_pruning_network.py`*
"""

def generate_report(results: list[dict], save_path: str = "report.md"):
    """Write a Markdown report summarising all experimental results."""

    # ── Results table ──────────────────────────────────────────────────────
    header = ("| λ (Lambda) | Test Accuracy (%) | Sparsity Level (%) | "
              "Active Weights | Pruned Weights |\n"
              "|:---:|:---:|:---:|:---:|:---:|")
    rows = []
    for res in results:
        total_w = len(res["gate_values"])
        active  = sum(1 for g in res["gate_values"] if g >= 1e-2)
        pruned  = total_w - active
        rows.append(
            f"| {res['lambd']} | {res['accuracy']:.2f} | "
            f"{res['sparsity_pct']:.1f} | "
            f"{active:,} / {total_w:,} | {pruned:,} |"
        )
    table_str = "\n".join([header] + rows)

    # ── Auto-generated analysis ────────────────────────────────────────────
    sorted_res = sorted(results, key=lambda r: r["lambd"])
    analysis_lines = []
    for i, res in enumerate(sorted_res):
        label = ["**Low λ**", "**Medium λ**", "**High λ**"][min(i, 2)]
        analysis_lines.append(
            f"- {label} (`λ={res['lambd']}`): "
            f"Test accuracy = **{res['accuracy']:.2f}%**, "
            f"Sparsity = **{res['sparsity_pct']:.1f}%**. "
        )
    # Add delta commentary
    if len(sorted_res) >= 2:
        acc_drop = sorted_res[0]["accuracy"] - sorted_res[-1]["accuracy"]
        sp_gain  = sorted_res[-1]["sparsity_pct"] - sorted_res[0]["sparsity_pct"]
        analysis_lines.append(
            f"\nIncreasing λ from {sorted_res[0]['lambd']} to {sorted_res[-1]['lambd']} "
            f"reduced test accuracy by **{acc_drop:.2f} pp** while increasing sparsity by "
            f"**{sp_gain:.1f} pp**. The results confirm that the L1 gate penalty "
            "effectively trades model capacity for compactness."
        )

    analysis_str = "\n".join(analysis_lines)

    # ── Write report ───────────────────────────────────────────────────────
    content = REPORT_TEMPLATE.format(
        results_table=table_str,
        analysis=analysis_str,
    )
    with open(save_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"  → Saved Markdown report to '{save_path}'")


# ─────────────────────────────────────────────
# Main entry point
# ─────────────────────────────────────────────

def main():
    # ── Configuration ──────────────────────────────────────────────────────
    LAMBDA_VALUES = [1e-5, 1e-4, 5e-4]   # low / medium / high sparsity
    EPOCHS        = 10                    # increase to 20+ for stronger results
    RESULTS_JSON  = "results.json"
    DIST_PLOT     = "gate_distributions.png"
    CURVES_PLOT   = "training_curves.png"
    TRADEOFF_PLOT = "sparsity_accuracy_tradeoff.png"
    REPORT_FILE   = "report.md"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'─'*55}")
    print(f"  Self-Pruning Neural Network — CIFAR-10")
    print(f"  Device  : {device}")
    print(f"  Lambdas : {LAMBDA_VALUES}")
    print(f"  Epochs  : {EPOCHS}")
    print(f"{'─'*55}")

    # ── Run experiments ────────────────────────────────────────────────────
    all_results = []
    for lambd in LAMBDA_VALUES:
        result = train_and_evaluate(lambd=lambd, epochs=EPOCHS,
                                    device=device, verbose=True)
        all_results.append(result)

    # ── Save raw results ───────────────────────────────────────────────────
    # Omit gate_values list from JSON (too large); keep summary only
    summary = [{k: v for k, v in r.items() if k != "gate_values"}
               for r in all_results]
    with open(RESULTS_JSON, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n  → Saved results summary to '{RESULTS_JSON}'")

    # ── Plots ──────────────────────────────────────────────────────────────
    print("\n  Generating plots …")
    plot_gate_distributions(all_results, save_path=DIST_PLOT)
    plot_training_curves(all_results,    save_path=CURVES_PLOT)
    plot_sparsity_accuracy_tradeoff(all_results, save_path=TRADEOFF_PLOT)

    # ── Report ─────────────────────────────────────────────────────────────
    generate_report(all_results, save_path=REPORT_FILE)

    # ── Final summary table ────────────────────────────────────────────────
    print(f"\n{'═'*55}")
    print(f"  {'λ':>10}  {'Accuracy':>10}  {'Sparsity':>10}")
    print(f"  {'-'*10}  {'-'*10}  {'-'*10}")
    for res in all_results:
        print(f"  {res['lambd']:>10}  "
              f"{res['accuracy']:>9.2f}%  "
              f"{res['sparsity_pct']:>9.1f}%")
    print(f"{'═'*55}")
    print("\n  All done! Outputs:")
    for fn in [RESULTS_JSON, DIST_PLOT, CURVES_PLOT, TRADEOFF_PLOT, REPORT_FILE]:
        print(f"    • {fn}")


if __name__ == "__main__":
    main()


───────────────────────────────────────────────────────
  Self-Pruning Neural Network — CIFAR-10
  Device  : cpu
  Lambdas : [1e-05, 0.0001, 0.0005]
  Epochs  : 10
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
  Training  λ = 1e-05   (10 epochs)  device=cpu
───────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Epoch   1/10  cls_loss=1.6311  total_loss=17.7335
  Epoch   2/10  cls_loss=1.4139  total_loss=17.1003
  Epoch   4/10  cls_loss=1.1910  total_loss=15.3509
  Epoch   6/10  cls_loss=0.9991  total_loss=12.7852
  Epoch   8/10  cls_loss=0.7399  total_loss=11.0819
  Epoch  10/10  cls_loss=0.5794  total_loss=10.3281

  ✓  λ=1e-05 → Accuracy=57.18%  Sparsity=0.0%

───────────────────────────────────────────────────────
  Training  λ = 0.0001   (10 epochs)  device=cpu
───────────────────────────────────────────────────────
  Epoch   1/10  cls_loss=1.6299  total_loss=162.4140
  Epoch   2/10  cls_loss=1.4091  total_loss=157.0434
  Epoch   4/10  cls_loss=1.1856  total_loss=134.5377
  Epoch   6/10  cls_loss=1.0031  total_loss=95.1645
  Epoch   8/10  cls_loss=0.8009  total_loss=71.4518
  Epoch  10/10  cls_loss=0.6915  total_loss=61.9074

  ✓  λ=0.0001 → Accuracy=57.30%  Sparsity=0.0%

───────────────────────────────────────────────────────
  Training  λ = 0.0005   (10 epochs)  device=cpu
──────────

ValueError: unexpected '{' in field name